# 2. Data augmentation

## 2.1. Setup and load processed data

Keep the processed inputs intact and create a separate dataframe for augmentation.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

teams_df = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
players_df = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")
games_df = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
event_types_df = pd.read_parquet(PROCESSED_DATA_DIR / "event_types.parquet")
events_df = pd.read_parquet(PROCESSED_DATA_DIR / "events.parquet")

# Work on a copy while retaining the processed events for comparison.
augmented_events_df = events_df.copy()

## 2.2. Zone attribution

In [21]:
# Zones are relative to the team credited with the event.
# Blue-line locations are included in the neutral zone.
x = augmented_events_df["x"]

augmented_events_df["event_team_zone"] = (
    pd.Series("NZ", index=x.index, dtype="string")
    .mask(x.lt(75).fillna(False), "DZ")
    .mask(x.gt(125).fillna(False), "OZ")
    .where(x.notna())
)

display(
    augmented_events_df.groupby(
        ["source_dataset", "event_team_zone"],
        dropna=False,
    ).size().unstack(fill_value=0)
)

event_team_zone,DZ,NZ,OZ
source_dataset,,,
NWHL,13715,8653,15193
Olympics 2022,6037,3055,5910
Womens,13342,7387,13378


## 2.3. Add game context

### Manpower and score context

In [22]:
augmented_events_df["is_5v5"] = (
    augmented_events_df["team_skaters"].eq(5)
    & augmented_events_df["opponent_skaters"].eq(5)
)

augmented_events_df["score_differential"] = (
    augmented_events_df["team_goals"]
    - augmented_events_df["opponent_goals"]
)

## 2.4. Model within-second timing

When `k` events have the same timestamp and different positions, add  `0, 1/k, ..., (k-1)/k` seconds to each event. Context rows share the preceding position's time. Events at 0:00 stays unchanged.

In [23]:
def model_event_clocks(events):
    # Events that represent puck positions.
    position_events = {
        "Puck Recovery", "Takeaway", "Faceoff Win", "Pass", "Pass Reception",
        "Incomplete Pass", "Shot", "Goal", "Dump In/Out", "Zone Entry",
    }

    # Order events and select position rows within each recorded second.
    ordered = events.sort_values("event_id", kind="stable").copy()
    keys = ["source_dataset", "game_id", "period", "clock_seconds"]
    candidate = ordered["event"].isin(position_events)
    anchors = ordered.loc[candidate].copy()

    # Consecutive positions from the same team share a timing group.
    previous = anchors.groupby(keys, sort=False, dropna=False)[["team_id", "x", "y"]].shift()
    same_position = (
        anchors["team_id"].eq(previous["team_id"])
        & anchors["x"].eq(previous["x"])
        & anchors["y"].eq(previous["y"])
    ).fillna(False)

    # Number position groups and spread them evenly within the second.
    groups = [anchors[key] for key in keys]
    position_group = (~same_position).groupby(groups, sort=False, dropna=False).cumsum()
    group_count = position_group.groupby(groups, sort=False, dropna=False).transform("max")
    offset = (position_group - 1) / group_count

    # Check that the recorded clocks and coordinates are usable.
    clock = anchors["clock_seconds"]
    valid = (
        clock.notna() & np.isfinite(clock) & clock.ge(0)
    ).fillna(False)

    # Subtract fractional offsets; keep 0:00 unchanged and invalid rows missing.
    anchors["modeled_clock_seconds"] = (
        clock - offset.where(clock.gt(0), 0)
    ).where(valid)

    # Context rows consume no timing slot. Attach them to the preceding
    # position in the same recorded second, or the original clock if leading.
    timeline = ordered["event_id"].map(anchors.set_index("event_id")["modeled_clock_seconds"])
    context_times = timeline.groupby(
        [ordered[key] for key in keys], sort=False, dropna=False
    ).ffill().fillna(ordered["clock_seconds"])
    ordered["modeled_clock_seconds"] = timeline.where(candidate, context_times).astype("Float64")

    # Map modeled times back to the original event order.
    result = events.copy()
    result["modeled_clock_seconds"] = result["event_id"].map(
        ordered.set_index("event_id")["modeled_clock_seconds"]
    ).astype("Float64")
    return result


# Apply modeled timing without changing recorded clocks.
augmented_events_df = model_event_clocks(augmented_events_df)

## 2.5. Define possession rules

Add `possession_team_id` (sequence owner) and `possession_id` (sequence identifier).

Following the [paper's sequence boundaries](../media/Playing%20Fast%20Not%20Loose.pdf), start a new sequence when possession or manpower changes, play restarts after a goal or at a faceoff, or the game/period changes. Isolate penalty-shot goals. Zone changes do not split these base sequences.

| Events | Ownership rule |
|---|---|
| Recovery, takeaway, faceoff win, pass/reception, incomplete pass, shot/goal, dump, any zone entry | Establish ownership for the event team at the recorded action. |
| Incomplete pass reception, penalty taken | Retain the previous owner; these rows do not establish control. |

Shots and unsuccessful releases do not immediately change ownership; the next control event determines it. Before any control evidence, use the event team as a provisional owner. These labels are inferred, unlike the paper's supplied possession labels.

### Calculate possession sequences

In [24]:
def infer_possession_context(events):
    control_events = {
        "Puck Recovery", "Takeaway", "Faceoff Win", "Pass",
        "Pass Reception", "Incomplete Pass", "Shot", "Goal",
        "Dump In/Out", "Zone Entry", "Penalty Shot Goal",
    }
    records = []
    owner = None
    sequence_id = 0
    previous_game = previous_period = previous_manpower = None
    after_goal = False

    for row in events.sort_values("event_id", kind="stable").itertuples(index=False):
        manpower = (
            row.team_skaters if row.is_home else row.opponent_skaters,
            row.opponent_skaters if row.is_home else row.team_skaters,
        )
        new_period = (
            row.game_id != previous_game or row.period != previous_period
        )
        restart = row.event in {"Faceoff Win", "Penalty Shot Goal"}
        new_sequence = (
            new_period or after_goal or restart
            or manpower != previous_manpower
        )

        # Restarts reset ownership; manpower changes alone preserve it.
        if new_period or after_goal or restart:
            owner = None

        if row.event in control_events:
            if owner is not None and owner != row.team_id:
                new_sequence = True
            owner = row.team_id
        elif owner is None:
            # Initial fallback until a control event establishes ownership.
            owner = row.team_id

        if new_sequence:
            sequence_id += 1
        records.append((row.event_id, owner, sequence_id))

        after_goal = row.event in {"Goal", "Penalty Shot Goal"}
        previous_game = row.game_id
        previous_period = row.period
        previous_manpower = manpower

    context = pd.DataFrame(
        records, columns=["event_id", "possession_team_id", "possession_id"]
    ).set_index("event_id")
    result = events.copy()
    for column in ["possession_team_id", "possession_id"]:
        result[column] = result["event_id"].map(context[column]).astype("Int64")
    return result


augmented_events_df = infer_possession_context(augmented_events_df)
print(f"Assigned {augmented_events_df['possession_id'].nunique():,} sequences.")


Assigned 16,046 sequences.


### Assess possession sequences

Check consistency and inspect sequence lengths. Recorded spans are not measured possession durations.

In [25]:
# Check that processing preserves events and produces consistent sequences.
p = augmented_events_df.sort_values("event_id", kind="stable").copy()
pd.testing.assert_frame_equal(
    p[events_df.columns].reset_index(drop=True),
    events_df.sort_values("event_id", kind="stable").reset_index(drop=True),
)
assert p[["possession_team_id", "possession_id"]].notna().all().all()
assert (p["possession_team_id"].eq(p["team_id"])
        | p["possession_team_id"].eq(p["opponent_team_id"])).all()

p["home_skaters_check"] = p["team_skaters"].where(p["is_home"], p["opponent_skaters"])
p["away_skaters_check"] = p["opponent_skaters"].where(p["is_home"], p["team_skaters"])
sequences = p.groupby("possession_id", sort=False)
for column in ["game_id", "period", "possession_team_id",
               "home_skaters_check", "away_skaters_check"]:
    assert sequences[column].nunique().eq(1).all(), column

control = p["event"].isin([
    "Puck Recovery", "Takeaway", "Faceoff Win", "Pass", "Pass Reception",
    "Incomplete Pass", "Shot", "Goal", "Dump In/Out", "Zone Entry", "Penalty Shot Goal",
])
assert p.loc[control, "possession_team_id"].eq(p.loc[control, "team_id"]).all()
starts = p["possession_id"].ne(p["possession_id"].shift())
assert starts.loc[p["event"].isin(["Faceoff Win", "Penalty Shot Goal"])].all()
after_goal = p["event"].shift().isin(["Goal", "Penalty Shot Goal"])
assert starts.loc[after_goal].all()
context = ~control & ~starts
assert p.loc[context, "possession_team_id"].eq(
    p["possession_team_id"].shift().loc[context]
).all()
assert not p.groupby("game_id")["period"].diff().lt(0).any()
assert not p.dropna(subset=["clock_seconds"]).groupby(
    ["game_id", "period"]
)["clock_seconds"].diff().gt(0).any()
print("Possession and sequence checks passed.")


Possession and sequence checks passed.


## 2.6. Prepare event transitions

Link consecutive puck positions within each possession sequence in a separate `transitions_df`. Keep final positions, recorded clocks, and modeled endpoint clocks. Distances, speeds, and terminal-event exclusions are deferred.

| Events | Position rule |
|---|---|
| Recovery, takeaway, faceoff win, pass/reception, incomplete pass, shot/goal, dump, any zone entry | Use the event team's recorded location when it owns the sequence. |
| Incomplete pass reception, penalty taken, penalty-shot goal | Exclude intended targets, contextual events, and the special-play goal. |

Missing endpoint values remain visible rather than connecting across a missing position.

### Build transitions

In [26]:
# Keep positions from the sequence owner, including its final position.
position_events = {
    "Puck Recovery", "Takeaway", "Faceoff Win", "Pass", "Pass Reception",
    "Incomplete Pass", "Shot", "Goal", "Dump In/Out", "Zone Entry",
}
positions = (
    augmented_events_df.loc[
        augmented_events_df["event"].isin(position_events)
        & augmented_events_df["team_id"].eq(augmented_events_df["possession_team_id"]),
        ["source_dataset", "game_id", "period", "possession_id", "possession_team_id",
         "event_id", "clock_seconds", "modeled_clock_seconds", "x", "y"],
    ]
    .sort_values("event_id", kind="stable")
    .reset_index(drop=True)
)

# Retain missing positions so we never silently bridge across them.
endpoint_columns = ["event_id", "clock_seconds", "modeled_clock_seconds", "x", "y"]
previous = positions.groupby("possession_id", sort=False)[endpoint_columns].shift()
has_previous = previous["event_id"].notna()

transitions_df = positions.loc[has_previous].rename(columns={
    column: f"end_{column}" for column in endpoint_columns
}).copy()
for column in endpoint_columns:
    transitions_df[f"start_{column}"] = previous.loc[has_previous, column]

transitions_df = transitions_df[[
    "source_dataset", "game_id", "period", "possession_id", "possession_team_id",
    "start_event_id", "end_event_id",
    "start_clock_seconds", "end_clock_seconds",
    "start_modeled_clock_seconds", "end_modeled_clock_seconds",
    "start_x", "start_y", "end_x", "end_y",
]].reset_index(drop=True)

print(f"Created {len(transitions_df):,} within-sequence transitions.")
display(transitions_df.head())


Created 63,298 within-sequence transitions.


,source_dataset,game_id,period,possession_id,possession_team_id,start_event_id,end_event_id,start_clock_seconds,end_clock_seconds,start_modeled_clock_seconds,end_modeled_clock_seconds,start_x,start_y,end_x,end_y
0,NWHL,3,1,1,1,1,2,1200.0,1198.0,1200.0,1198.0,100.0,43.0,107.0,40.0
1,NWHL,3,1,1,1,2,3,1198.0,1197.0,1198.0,1197.0,107.0,40.0,125.0,28.0
2,NWHL,3,1,1,1,3,4,1197.0,1195.0,1197.0,1195.0,125.0,28.0,131.0,28.0
3,NWHL,3,1,2,1,5,6,1193.0,1192.0,1193.0,1192.0,169.0,21.0,159.0,26.0
4,NWHL,3,1,3,5,7,8,1190.0,1188.0,1190.0,1188.0,101.0,31.0,125.0,6.0


### Assess transitions

In [27]:
# Verify endpoint relationships and retain the final position of each sequence.
lookup = augmented_events_df.set_index("event_id")
assert transitions_df["start_event_id"].lt(transitions_df["end_event_id"]).all()
for endpoint in ["start", "end"]:
    ids = transitions_df[f"{endpoint}_event_id"]
    for column in ["game_id", "period", "possession_id", "possession_team_id", "source_dataset"]:
        assert ids.map(lookup[column]).eq(transitions_df[column]).all(), column
    assert ids.map(lookup["team_id"]).eq(transitions_df["possession_team_id"]).all()
    pd.testing.assert_series_equal(
        ids.map(lookup["modeled_clock_seconds"]).reset_index(drop=True),
        transitions_df[f"{endpoint}_modeled_clock_seconds"].reset_index(drop=True),
        check_names=False,
    )

sizes = positions.groupby("possession_id").size()
assert len(transitions_df) == int((sizes - 1).sum())
last_positions = positions.groupby("possession_id", sort=False).tail(1)
last_positions = last_positions.loc[last_positions["possession_id"].map(sizes).ge(2)]
assert last_positions["event_id"].isin(transitions_df["end_event_id"]).all()

# Diagnostics stay separate from the transitions and augmented event tables.
transition_check = transitions_df.copy()
transition_check["elapsed_seconds"] = (
    transition_check["start_clock_seconds"] - transition_check["end_clock_seconds"]
)
assert transition_check["elapsed_seconds"].dropna().ge(0).all()
transition_check["modeled_elapsed_seconds"] = (
    transition_check["start_modeled_clock_seconds"] - transition_check["end_modeled_clock_seconds"]
)
assert transition_check["modeled_elapsed_seconds"].dropna().ge(0).all()
transition_check["modeled_zero"] = transition_check["modeled_elapsed_seconds"].eq(0)
transition_check["same_second"] = transition_check["elapsed_seconds"].eq(0)
transition_check["positive_time"] = transition_check["elapsed_seconds"].gt(0)
transition_check["missing_clock"] = transition_check[[
    "start_clock_seconds", "end_clock_seconds"
]].isna().any(axis=1)
transition_check["missing_position"] = transition_check[[
    "start_x", "start_y", "end_x", "end_y"
]].isna().any(axis=1)

print("Transition checks passed; tied timestamps and final positions are retained.")
display(transition_check.groupby("source_dataset").agg(
    transitions=("end_event_id", "size"),
    sequences=("possession_id", "nunique"),
    positive_time=("positive_time", "sum"),
    same_second=("same_second", "sum"),
    modeled_zero=("modeled_zero", "sum"),
    missing_clock=("missing_clock", "sum"),
    missing_position=("missing_position", "sum"),
))


Transition checks passed; tied timestamps and final positions are retained.


,transitions,sequences,positive_time,same_second,modeled_zero,missing_clock,missing_position
source_dataset,,,,,,,
NWHL,26777,6473,15066,11711,3778,0,0
Olympics 2022,11258,2413,6122,5136,1108,0,0
Womens,25263,5440,13869,11394,3313,0,0


## 2.7. Build possession context

Summarize every possession, including single-event sequences and sequences without movement transitions. Boundaries and outcome use event order, not modeled timing. Take scores and manpower from the last event in home/away perspective. If a sequence contains a `Goal` or `Penalty Shot Goal`, increment the possession owner's score once, matching the current database importer.

Work on a separate copy so the augmented events export retains its 27-column schema. Recorded clock spans are not modeled possession durations.

In [28]:
def build_possession_context(events, games):
    ordered = events.sort_values("event_id", kind="stable").copy()
    grouped = ordered.groupby("possession_id", sort=False)
    context_keys = ["game_id", "period", "possession_team_id", "source_dataset"]
    if not grouped[context_keys].nunique(dropna=False).eq(1).all().all():
        raise ValueError("Possessions have inconsistent game, period, owner, or source.")

    ordered["home_score"] = ordered["team_goals"].where(ordered["is_home"], ordered["opponent_goals"])
    ordered["away_score"] = ordered["opponent_goals"].where(ordered["is_home"], ordered["team_goals"])
    ordered["home_skaters"] = ordered["team_skaters"].where(ordered["is_home"], ordered["opponent_skaters"])
    ordered["away_skaters"] = ordered["opponent_skaters"].where(ordered["is_home"], ordered["team_skaters"])
    ordered["contains_goal"] = ordered["event"].isin(["Goal", "Penalty Shot Goal"])
    context = ordered.groupby("possession_id", sort=False).agg(
        game_id=("game_id", "first"),
        period=("period", "first"),
        possession_team_id=("possession_team_id", "first"),
        source_dataset=("source_dataset", "first"),
        start_event_id=("event_id", "first"),
        end_event_id=("event_id", "last"),
        start_clock_seconds=("clock_seconds", "first"),
        end_clock_seconds=("clock_seconds", "last"),
        event_count=("event_id", "size"),
        outcome=("event", "last"),
        home_score=("home_score", "last"),
        away_score=("away_score", "last"),
        home_skaters=("home_skaters", "last"),
        away_skaters=("away_skaters", "last"),
        contains_goal=("contains_goal", "any"),
    ).reset_index()

    game_lookup = games.set_index("game_id")
    home = context["game_id"].map(game_lookup["home_team_id"])
    away = context["game_id"].map(game_lookup["away_team_id"])
    if not (context["possession_team_id"].eq(home) | context["possession_team_id"].eq(away)).all():
        raise ValueError("A possession owner is not a participating team.")
    context["home_score"] += (context["contains_goal"] & context["possession_team_id"].eq(home)).astype("Int64")
    context["away_score"] += (context["contains_goal"] & context["possession_team_id"].eq(away)).astype("Int64")

    integers = ["possession_id", "game_id", "period", "possession_team_id",
                "start_event_id", "end_event_id", "event_count",
                "home_score", "away_score", "home_skaters", "away_skaters"]
    context[integers] = context[integers].astype("Int64")
    context[["start_clock_seconds", "end_clock_seconds"]] = context[
        ["start_clock_seconds", "end_clock_seconds"]
    ].astype("Float64")
    context[["source_dataset", "outcome"]] = context[["source_dataset", "outcome"]].astype("string")
    context["contains_goal"] = context["contains_goal"].astype("boolean")
    return context.sort_values("possession_id", kind="stable").reset_index(drop=True)


possession_context_df = build_possession_context(augmented_events_df, games_df)
print(f"Built context for {len(possession_context_df):,} possessions.")
display(possession_context_df.head())

Built context for 16,046 possessions.


,possession_id,game_id,period,possession_team_id,source_dataset,start_event_id,end_event_id,start_clock_seconds,end_clock_seconds,event_count,outcome,home_score,away_score,home_skaters,away_skaters,contains_goal
0,1,3,1,1,NWHL,1,4,1200.0,1195.0,4,Shot,0,0,5,5,False
1,2,3,1,1,NWHL,5,6,1193.0,1192.0,2,Puck Recovery,0,0,5,5,False
2,3,3,1,5,NWHL,7,12,1190.0,1183.0,6,Pass Reception,0,0,5,5,False
3,4,3,1,1,NWHL,13,16,1180.0,1177.0,4,Dump In/Out,0,0,5,5,False
4,5,3,1,5,NWHL,17,19,1176.0,1174.0,3,Incomplete Pass Reception,0,0,5,5,False


## 2.8. Export augmented files

Export final augmented events, intermediate event transitions, and possession context to distinct Parquet files in `data/processed`, preserving notebook 01 outputs. Verify each export by reading it back.

In [29]:
export_tables = {
    "events_augmented": augmented_events_df,
    "event_transitions": transitions_df,
    "possession_context": possession_context_df,
}

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
export_summary = []
for name, df in export_tables.items():
    path = PROCESSED_DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, engine="pyarrow", compression="snappy", index=False)
    saved = pd.read_parquet(path, engine="pyarrow")
    pd.testing.assert_frame_equal(saved, df.reset_index(drop=True), check_exact=True)
    export_summary.append({
        "table": name, "rows": len(saved), "columns": len(saved.columns), "file": str(path),
    })

print("Exported and verified all three Parquet files.")
display(pd.DataFrame(export_summary))

Exported and verified all three Parquet files.


,table,rows,columns,file
0,events_augmented,86670,27,..\data\processed\events_augmented.parquet
1,event_transitions,63298,15,..\data\processed\event_transitions.parquet
2,possession_context,16046,16,..\data\processed\possession_context.parquet
